# 01 - Data Collection and Entity Resolution

Goal: collect public daily weather data for Southeast Asia rainfall forecasting, then resolve records from multiple sources to the same real-world location entities.

Collection choices:
- Use only public no-key APIs: NASA POWER Daily API and Open-Meteo Historical Weather API.
- Do not read, print, or store secrets/tokens.
- Save a small, clearly named group of files in `data/raw/`.
- Store raw API payloads in one JSONL file instead of creating many per-location JSON files.

## Output Files

This notebook writes five synchronized files:

- `sea_rainfall_daily_2020_2025_raw_api_responses.jsonl`: raw payloads, one line per source-location request.
- `sea_rainfall_daily_2020_2025_source_observations.csv`: standardized daily observations before cross-source merging.
- `sea_rainfall_daily_2020_2025_entity_resolved.csv`: one row per canonical location-date after entity resolution.
- `sea_rainfall_daily_2020_2025_entity_registry.csv`: canonical entity table and the matching coordinates returned by each source.
- `sea_rainfall_daily_2020_2025_manifest.json`: data provenance, row counts, checksums, and quality summary.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import time
import urllib.parse
import urllib.request
from urllib.error import HTTPError, URLError

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE_ISO = '2020-01-01'
END_DATE_ISO = '2025-12-31'
START_DATE_NASA = START_DATE_ISO.replace('-', '')
END_DATE_NASA = END_DATE_ISO.replace('-', '')
BASE_STEM = 'sea_rainfall_daily_2020_2025'

RAW_RESPONSES_PATH = RAW_DIR / f'{BASE_STEM}_raw_api_responses.jsonl'
SOURCE_OBSERVATIONS_PATH = RAW_DIR / f'{BASE_STEM}_source_observations.csv'
ENTITY_RESOLVED_PATH = RAW_DIR / f'{BASE_STEM}_entity_resolved.csv'
ENTITY_REGISTRY_PATH = RAW_DIR / f'{BASE_STEM}_entity_registry.csv'
MANIFEST_PATH = RAW_DIR / f'{BASE_STEM}_manifest.json'

NASA_POWER_DAILY_URL = 'https://power.larc.nasa.gov/api/temporal/daily/point'
OPEN_METEO_ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'

NASA_PARAMETERS = [
    'PRECTOTCORR',
    'T2M',
    'T2M_MAX',
    'T2M_MIN',
    'RH2M',
    'WS2M',
    'PS',
]

OPEN_METEO_DAILY_VARIABLES = [
    'precipitation_sum',
    'temperature_2m_mean',
    'temperature_2m_max',
    'temperature_2m_min',
    'relative_humidity_2m_mean',
    'wind_speed_10m_mean',
    'surface_pressure_mean',
]

# Canonical real-world location entities. Records from different APIs are mapped to these IDs.
LOCATIONS = [
    {'entity_id': 'SEA_VN_HANOI', 'country': 'Vietnam', 'location_name': 'Hanoi', 'canonical_latitude': 21.0278, 'canonical_longitude': 105.8342},
    {'entity_id': 'SEA_VN_HO_CHI_MINH', 'country': 'Vietnam', 'location_name': 'Ho Chi Minh City', 'canonical_latitude': 10.8231, 'canonical_longitude': 106.6297},
    {'entity_id': 'SEA_TH_BANGKOK', 'country': 'Thailand', 'location_name': 'Bangkok', 'canonical_latitude': 13.7563, 'canonical_longitude': 100.5018},
    {'entity_id': 'SEA_KH_PHNOM_PENH', 'country': 'Cambodia', 'location_name': 'Phnom Penh', 'canonical_latitude': 11.5564, 'canonical_longitude': 104.9282},
    {'entity_id': 'SEA_LA_VIENTIANE', 'country': 'Laos', 'location_name': 'Vientiane', 'canonical_latitude': 17.9757, 'canonical_longitude': 102.6331},
    {'entity_id': 'SEA_MM_YANGON', 'country': 'Myanmar', 'location_name': 'Yangon', 'canonical_latitude': 16.8409, 'canonical_longitude': 96.1735},
    {'entity_id': 'SEA_MY_KUALA_LUMPUR', 'country': 'Malaysia', 'location_name': 'Kuala Lumpur', 'canonical_latitude': 3.1390, 'canonical_longitude': 101.6869},
    {'entity_id': 'SEA_SG_SINGAPORE', 'country': 'Singapore', 'location_name': 'Singapore', 'canonical_latitude': 1.3521, 'canonical_longitude': 103.8198},
    {'entity_id': 'SEA_ID_JAKARTA', 'country': 'Indonesia', 'location_name': 'Jakarta', 'canonical_latitude': -6.2088, 'canonical_longitude': 106.8456},
    {'entity_id': 'SEA_PH_MANILA', 'country': 'Philippines', 'location_name': 'Manila', 'canonical_latitude': 14.5995, 'canonical_longitude': 120.9842},
    {'entity_id': 'SEA_BN_BANDAR_SERI_BEGAWAN', 'country': 'Brunei', 'location_name': 'Bandar Seri Begawan', 'canonical_latitude': 4.9031, 'canonical_longitude': 114.9398},
    {'entity_id': 'SEA_TL_DILI', 'country': 'Timor-Leste', 'location_name': 'Dili', 'canonical_latitude': -8.5569, 'canonical_longitude': 125.5603},
]

STANDARD_VARIABLES = [
    'precipitation_mm',
    'temp_mean_c',
    'temp_max_c',
    'temp_min_c',
    'relative_humidity_pct',
    'wind_speed_ms',
    'surface_pressure_kpa',
]

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data directory: {RAW_DIR}')
print(f'Date range: {START_DATE_ISO} to {END_DATE_ISO}')
print(f'Canonical entities: {len(LOCATIONS)}')

Project root: D:\DS\Project
Raw data directory: D:\DS\Project\data\raw
Date range: 2020-01-01 to 2025-12-31
Canonical entities: 12


In [2]:
def build_url(base_url, query):
    return f'{base_url}?{urllib.parse.urlencode(query)}'


def fetch_json(url, timeout=90, max_retries=4, pause_seconds=2):
    request = urllib.request.Request(
        url,
        headers={'User-Agent': 'ds108-rainfall-collection/2.0 no-api-key'},
    )
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                payload = response.read().decode('utf-8')
            return json.loads(payload)
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as error:
            last_error = error
            if attempt == max_retries:
                break
            time.sleep(pause_seconds * attempt)
    raise RuntimeError(f'Failed to fetch JSON after {max_retries} attempts: {last_error}')


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    return radius_km * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def resolve_entity(source_latitude, source_longitude, max_distance_km=50):
    candidates = []
    for location in LOCATIONS:
        distance_km = haversine_km(
            source_latitude,
            source_longitude,
            location['canonical_latitude'],
            location['canonical_longitude'],
        )
        candidates.append((distance_km, location))

    distance_km, best_location = min(candidates, key=lambda item: item[0])
    if distance_km > max_distance_km:
        raise ValueError(
            f'No canonical entity within {max_distance_km} km for source coordinate '
            f'({source_latitude}, {source_longitude}); nearest={best_location["entity_id"]}, distance={distance_km:.2f} km'
        )

    return best_location, distance_km


def none_if_missing(value):
    if value in (-999, -999.0, None):
        return None
    return value


def build_nasa_url(location):
    query = {
        'parameters': ','.join(NASA_PARAMETERS),
        'community': 'AG',
        'longitude': location['canonical_longitude'],
        'latitude': location['canonical_latitude'],
        'start': START_DATE_NASA,
        'end': END_DATE_NASA,
        'format': 'JSON',
    }
    return build_url(NASA_POWER_DAILY_URL, query)


def build_open_meteo_url(location):
    query = {
        'latitude': location['canonical_latitude'],
        'longitude': location['canonical_longitude'],
        'start_date': START_DATE_ISO,
        'end_date': END_DATE_ISO,
        'daily': ','.join(OPEN_METEO_DAILY_VARIABLES),
        'timezone': 'UTC',
    }
    return build_url(OPEN_METEO_ARCHIVE_URL, query)

In [3]:
def parse_nasa_power(payload):
    geometry = payload.get('geometry', {})
    coordinates = geometry.get('coordinates') or []
    if len(coordinates) >= 2:
        source_longitude = float(coordinates[0])
        source_latitude = float(coordinates[1])
    else:
        header = payload.get('header', {})
        source_latitude = float(header.get('latitude'))
        source_longitude = float(header.get('longitude'))

    entity, entity_distance_km = resolve_entity(source_latitude, source_longitude)
    parameters = payload.get('properties', {}).get('parameter', {})
    if not parameters:
        raise ValueError('NASA POWER response does not contain parameter data')

    source_entity_key = f'NASA_POWER_DAILY:{round(source_latitude, 4)}:{round(source_longitude, 4)}'
    records = []
    all_dates = sorted({day for values in parameters.values() for day in values.keys()})

    for day in all_dates:
        records.append({
            'source': 'NASA_POWER_DAILY',
            'source_entity_key': source_entity_key,
            'entity_id': entity['entity_id'],
            'country': entity['country'],
            'location_name': entity['location_name'],
            'canonical_latitude': entity['canonical_latitude'],
            'canonical_longitude': entity['canonical_longitude'],
            'source_latitude': source_latitude,
            'source_longitude': source_longitude,
            'entity_distance_km': round(entity_distance_km, 4),
            'entity_resolution_method': 'nearest_canonical_coordinate_within_50km',
            'date': pd.to_datetime(day, format='%Y%m%d').date().isoformat(),
            'precipitation_mm': none_if_missing(parameters.get('PRECTOTCORR', {}).get(day)),
            'temp_mean_c': none_if_missing(parameters.get('T2M', {}).get(day)),
            'temp_max_c': none_if_missing(parameters.get('T2M_MAX', {}).get(day)),
            'temp_min_c': none_if_missing(parameters.get('T2M_MIN', {}).get(day)),
            'relative_humidity_pct': none_if_missing(parameters.get('RH2M', {}).get(day)),
            'wind_speed_ms': none_if_missing(parameters.get('WS2M', {}).get(day)),
            'surface_pressure_kpa': none_if_missing(parameters.get('PS', {}).get(day)),
        })

    metadata = {
        'source': 'NASA_POWER_DAILY',
        'source_entity_key': source_entity_key,
        'entity_id': entity['entity_id'],
        'source_latitude': source_latitude,
        'source_longitude': source_longitude,
        'entity_distance_km': round(entity_distance_km, 4),
        'entity_resolution_method': 'nearest_canonical_coordinate_within_50km',
    }
    return records, metadata


def parse_open_meteo(payload):
    source_latitude = float(payload['latitude'])
    source_longitude = float(payload['longitude'])
    entity, entity_distance_km = resolve_entity(source_latitude, source_longitude)
    daily = payload.get('daily', {})
    dates = daily.get('time', [])
    if not dates:
        raise ValueError('Open-Meteo response does not contain daily time data')

    source_entity_key = f'OPEN_METEO_ARCHIVE:{round(source_latitude, 4)}:{round(source_longitude, 4)}'
    records = []

    for index, day in enumerate(dates):
        wind_kmh = none_if_missing(daily.get('wind_speed_10m_mean', [None] * len(dates))[index])
        pressure_hpa = none_if_missing(daily.get('surface_pressure_mean', [None] * len(dates))[index])
        records.append({
            'source': 'OPEN_METEO_ARCHIVE',
            'source_entity_key': source_entity_key,
            'entity_id': entity['entity_id'],
            'country': entity['country'],
            'location_name': entity['location_name'],
            'canonical_latitude': entity['canonical_latitude'],
            'canonical_longitude': entity['canonical_longitude'],
            'source_latitude': source_latitude,
            'source_longitude': source_longitude,
            'entity_distance_km': round(entity_distance_km, 4),
            'entity_resolution_method': 'nearest_canonical_coordinate_within_50km',
            'date': day,
            'precipitation_mm': none_if_missing(daily.get('precipitation_sum', [None] * len(dates))[index]),
            'temp_mean_c': none_if_missing(daily.get('temperature_2m_mean', [None] * len(dates))[index]),
            'temp_max_c': none_if_missing(daily.get('temperature_2m_max', [None] * len(dates))[index]),
            'temp_min_c': none_if_missing(daily.get('temperature_2m_min', [None] * len(dates))[index]),
            'relative_humidity_pct': none_if_missing(daily.get('relative_humidity_2m_mean', [None] * len(dates))[index]),
            'wind_speed_ms': None if wind_kmh is None else round(wind_kmh / 3.6, 4),
            'surface_pressure_kpa': None if pressure_hpa is None else round(pressure_hpa / 10, 4),
        })

    metadata = {
        'source': 'OPEN_METEO_ARCHIVE',
        'source_entity_key': source_entity_key,
        'entity_id': entity['entity_id'],
        'source_latitude': source_latitude,
        'source_longitude': source_longitude,
        'entity_distance_km': round(entity_distance_km, 4),
        'entity_resolution_method': 'nearest_canonical_coordinate_within_50km',
        'timezone': payload.get('timezone'),
        'elevation_m': payload.get('elevation'),
    }
    return records, metadata

In [4]:
def build_entity_registry(source_metadata):
    rows = []
    for location in LOCATIONS:
        row = dict(location)
        matches = [item for item in source_metadata if item['entity_id'] == location['entity_id']]
        for match in matches:
            prefix = 'nasa_power' if match['source'] == 'NASA_POWER_DAILY' else 'open_meteo'
            row[f'{prefix}_source_entity_key'] = match['source_entity_key']
            row[f'{prefix}_source_latitude'] = match['source_latitude']
            row[f'{prefix}_source_longitude'] = match['source_longitude']
            row[f'{prefix}_distance_to_canonical_km'] = match['entity_distance_km']
            row[f'{prefix}_resolution_method'] = match['entity_resolution_method']
        rows.append(row)
    return pd.DataFrame(rows).sort_values('entity_id').reset_index(drop=True)


def build_entity_resolved_table(source_df):
    id_columns = ['entity_id', 'date']
    metadata_columns = ['country', 'location_name', 'canonical_latitude', 'canonical_longitude']

    nasa = source_df[source_df['source'] == 'NASA_POWER_DAILY'][id_columns + STANDARD_VARIABLES]
    nasa = nasa.rename(columns={column: f'nasa_power_{column}' for column in STANDARD_VARIABLES})

    open_meteo = source_df[source_df['source'] == 'OPEN_METEO_ARCHIVE'][id_columns + STANDARD_VARIABLES]
    open_meteo = open_meteo.rename(columns={column: f'open_meteo_{column}' for column in STANDARD_VARIABLES})

    metadata = source_df[id_columns + metadata_columns].drop_duplicates(id_columns)
    resolved = metadata.merge(nasa, on=id_columns, how='left').merge(open_meteo, on=id_columns, how='left')

    precip_columns = ['nasa_power_precipitation_mm', 'open_meteo_precipitation_mm']
    resolved['precipitation_mm_mean_two_sources'] = resolved[precip_columns].mean(axis=1, skipna=True)
    resolved['precipitation_mm_abs_diff'] = (
        resolved['nasa_power_precipitation_mm'] - resolved['open_meteo_precipitation_mm']
    ).abs()

    return resolved.sort_values(['entity_id', 'date']).reset_index(drop=True)


def summarize_quality(source_df, resolved_df):
    expected_dates = pd.date_range(START_DATE_ISO, END_DATE_ISO, freq='D')
    expected_rows_per_source = len(LOCATIONS) * len(expected_dates)

    duplicate_source_rows = int(source_df.duplicated(['source', 'entity_id', 'date']).sum())
    duplicate_resolved_rows = int(resolved_df.duplicated(['entity_id', 'date']).sum())
    negative_precipitation_rows = int((source_df['precipitation_mm'] < 0).sum())

    if duplicate_source_rows:
        raise ValueError(f'Duplicate source/entity/date rows: {duplicate_source_rows}')
    if duplicate_resolved_rows:
        raise ValueError(f'Duplicate entity/date rows after resolution: {duplicate_resolved_rows}')
    if negative_precipitation_rows:
        raise ValueError(f'Negative precipitation rows: {negative_precipitation_rows}')

    source_counts = source_df.groupby('source').size().to_dict()
    source_missing_rate = source_df.groupby('source')[STANDARD_VARIABLES].apply(lambda frame: frame.isna().mean()).round(6)
    resolved_missing_rate = resolved_df.isna().mean().sort_values(ascending=False).round(6)

    return {
        'expected_daily_dates': len(expected_dates),
        'expected_rows_per_source': expected_rows_per_source,
        'actual_rows_by_source': {key: int(value) for key, value in source_counts.items()},
        'source_duplicate_rows': duplicate_source_rows,
        'resolved_duplicate_rows': duplicate_resolved_rows,
        'negative_precipitation_rows': negative_precipitation_rows,
        'source_missing_rate': source_missing_rate.reset_index().to_dict(orient='records'),
        'resolved_top_missing_rate': resolved_missing_rate.head(20).to_dict(),
    }

In [5]:
all_records = []
source_metadata = []
raw_response_rows = []
collection_started_at = datetime.now(timezone.utc).isoformat(timespec='seconds')

for index, location in enumerate(LOCATIONS, start=1):
    print(f'[{index:02d}/{len(LOCATIONS):02d}] Collecting {location["entity_id"]}')

    nasa_url = build_nasa_url(location)
    nasa_payload = fetch_json(nasa_url)
    nasa_records, nasa_metadata = parse_nasa_power(nasa_payload)
    all_records.extend(nasa_records)
    source_metadata.append(nasa_metadata)
    raw_response_rows.append({
        'source': 'NASA_POWER_DAILY',
        'requested_entity_id': location['entity_id'],
        'request_url_without_secret': nasa_url,
        'payload': nasa_payload,
    })

    time.sleep(0.4)

    open_meteo_url = build_open_meteo_url(location)
    open_meteo_payload = fetch_json(open_meteo_url)
    open_meteo_records, open_meteo_metadata = parse_open_meteo(open_meteo_payload)
    all_records.extend(open_meteo_records)
    source_metadata.append(open_meteo_metadata)
    raw_response_rows.append({
        'source': 'OPEN_METEO_ARCHIVE',
        'requested_entity_id': location['entity_id'],
        'request_url_without_secret': open_meteo_url,
        'payload': open_meteo_payload,
    })

    time.sleep(0.8)

source_df = pd.DataFrame.from_records(all_records)
source_df['date'] = pd.to_datetime(source_df['date'])
source_df = source_df.sort_values(['source', 'entity_id', 'date']).reset_index(drop=True)

entity_registry_df = build_entity_registry(source_metadata)
resolved_df = build_entity_resolved_table(source_df)
quality_summary = summarize_quality(source_df, resolved_df)

with RAW_RESPONSES_PATH.open('w', encoding='utf-8') as file:
    for row in raw_response_rows:
        file.write(json.dumps(row, sort_keys=True) + '\n')

source_df.to_csv(SOURCE_OBSERVATIONS_PATH, index=False)
entity_registry_df.to_csv(ENTITY_REGISTRY_PATH, index=False)
resolved_df.to_csv(ENTITY_RESOLVED_PATH, index=False)

manifest = {
    'dataset_name': 'Multi-source Southeast Asia daily rainfall dataset 2020-2025',
    'collection_started_at_utc': collection_started_at,
    'collection_finished_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'created_by_notebook': 'notebooks/01_data_collection.ipynb',
    'secret_policy': 'No API key or token is required, read, printed, or stored.',
    'date_range': {'start': START_DATE_ISO, 'end': END_DATE_ISO, 'frequency': 'daily'},
    'entity_resolution': {
        'canonical_key': 'entity_id',
        'merge_key': ['entity_id', 'date'],
        'method': 'nearest canonical coordinate within 50 km for each source-returned coordinate',
        'purpose': 'Map NASA POWER and Open-Meteo records describing the same city/location to one canonical entity.',
    },
    'sources': [
        {
            'name': 'NASA POWER Daily API',
            'base_url': NASA_POWER_DAILY_URL,
            'documentation': 'https://power.larc.nasa.gov/docs/services/api/temporal/daily/',
            'requires_api_key': False,
        },
        {
            'name': 'Open-Meteo Historical Weather API',
            'base_url': OPEN_METEO_ARCHIVE_URL,
            'documentation': 'https://open-meteo.com/en/docs/historical-weather-api',
            'requires_api_key': False,
        },
    ],
    'canonical_entities': len(LOCATIONS),
    'standard_variables': STANDARD_VARIABLES,
    'row_counts': {
        'raw_api_requests': len(raw_response_rows),
        'source_observations': int(len(source_df)),
        'entity_resolved_rows': int(len(resolved_df)),
        'entity_registry_rows': int(len(entity_registry_df)),
    },
    'quality_summary': quality_summary,
    'files': [],
}

for path in [RAW_RESPONSES_PATH, SOURCE_OBSERVATIONS_PATH, ENTITY_RESOLVED_PATH, ENTITY_REGISTRY_PATH]:
    manifest['files'].append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'sha256': sha256_file(path),
    })

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print('Data collection completed')
print(f'Raw responses: {RAW_RESPONSES_PATH}')
print(f'Source observations: {SOURCE_OBSERVATIONS_PATH} | rows={len(source_df):,}')
print(f'Entity-resolved table: {ENTITY_RESOLVED_PATH} | rows={len(resolved_df):,}')
print(f'Entity registry: {ENTITY_REGISTRY_PATH} | rows={len(entity_registry_df):,}')
print(f'Manifest: {MANIFEST_PATH}')

[01/12] Collecting SEA_VN_HANOI
[02/12] Collecting SEA_VN_HO_CHI_MINH
[03/12] Collecting SEA_TH_BANGKOK
[04/12] Collecting SEA_KH_PHNOM_PENH
[05/12] Collecting SEA_LA_VIENTIANE
[06/12] Collecting SEA_MM_YANGON
[07/12] Collecting SEA_MY_KUALA_LUMPUR
[08/12] Collecting SEA_SG_SINGAPORE
[09/12] Collecting SEA_ID_JAKARTA
[10/12] Collecting SEA_PH_MANILA
[11/12] Collecting SEA_BN_BANDAR_SERI_BEGAWAN
[12/12] Collecting SEA_TL_DILI
Data collection completed
Raw responses: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_raw_api_responses.jsonl
Source observations: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_source_observations.csv | rows=52,608
Entity-resolved table: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_entity_resolved.csv | rows=26,304
Entity registry: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_entity_registry.csv | rows=12
Manifest: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_manifest.json


In [6]:
print('Quality summary')
print(json.dumps({
    'expected_daily_dates': quality_summary['expected_daily_dates'],
    'expected_rows_per_source': quality_summary['expected_rows_per_source'],
    'actual_rows_by_source': quality_summary['actual_rows_by_source'],
    'source_duplicate_rows': quality_summary['source_duplicate_rows'],
    'resolved_duplicate_rows': quality_summary['resolved_duplicate_rows'],
    'negative_precipitation_rows': quality_summary['negative_precipitation_rows'],
}, indent=2))

print('\nEntity registry preview')
print(entity_registry_df.head().to_string(index=False))

print('\nResolved table preview')
print(resolved_df.head().to_string(index=False))

Quality summary
{
  "expected_daily_dates": 2192,
  "expected_rows_per_source": 26304,
  "actual_rows_by_source": {
    "NASA_POWER_DAILY": 26304,
    "OPEN_METEO_ARCHIVE": 26304
  },
  "source_duplicate_rows": 0,
  "resolved_duplicate_rows": 0,
  "negative_precipitation_rows": 0
}

Entity registry preview
                 entity_id   country       location_name  canonical_latitude  canonical_longitude    nasa_power_source_entity_key  nasa_power_source_latitude  nasa_power_source_longitude  nasa_power_distance_to_canonical_km             nasa_power_resolution_method        open_meteo_source_entity_key  open_meteo_source_latitude  open_meteo_source_longitude  open_meteo_distance_to_canonical_km             open_meteo_resolution_method
SEA_BN_BANDAR_SERI_BEGAWAN    Brunei Bandar Seri Begawan              4.9031             114.9398   NASA_POWER_DAILY:4.903:114.94                       4.903                      114.940                               0.0248 nearest_canonical_coordinate_wit

## Notes

- This is still the raw/bronze-layer notebook. Later notebooks should handle feature engineering, train/test splitting, imputation, and modeling-ready datasets in `data/processed/`.
- The two sources are not forced to agree. The resolved table keeps each source's values and adds comparison columns for rainfall quality checks.
- Open-Meteo returns wind speed in km/h and pressure in hPa; these are standardized to m/s and kPa to match the shared schema.

## 01 Data Collection Report Output

This section writes a compact report for Notebook 01. It does not call the APIs again; it reads the raw files and manifest already created by the collection pipeline, then exports documentation tables and a Markdown report to `reports/01_data_collection_report/`.

In [7]:
# Build Notebook 01 report artifacts from already-collected raw outputs.
# This cell is intentionally placed inside notebook 01 so the report is reproducible from the notebook itself.
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

try:
    BASE_STEM
except NameError:
    BASE_STEM = 'sea_rainfall_daily_2020_2025'

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports' / '01_data_collection_report'
REPORT_TABLE_DIR = REPORT_DIR / 'tables'
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

RAW_RESPONSES_PATH = RAW_DIR / f'{BASE_STEM}_raw_api_responses.jsonl'
SOURCE_OBSERVATIONS_PATH = RAW_DIR / f'{BASE_STEM}_source_observations.csv'
ENTITY_RESOLVED_PATH = RAW_DIR / f'{BASE_STEM}_entity_resolved.csv'
ENTITY_REGISTRY_PATH = RAW_DIR / f'{BASE_STEM}_entity_registry.csv'
MANIFEST_PATH = RAW_DIR / f'{BASE_STEM}_manifest.json'

required_report_inputs = [
    RAW_RESPONSES_PATH,
    SOURCE_OBSERVATIONS_PATH,
    ENTITY_RESOLVED_PATH,
    ENTITY_REGISTRY_PATH,
    MANIFEST_PATH,
]
missing_report_inputs = [str(path) for path in required_report_inputs if not path.exists()]
if missing_report_inputs:
    raise FileNotFoundError('Run the data collection cells before building the report. Missing:\n' + '\n'.join(missing_report_inputs))

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
source_report_df = pd.read_csv(SOURCE_OBSERVATIONS_PATH, parse_dates=['date'])
resolved_report_df = pd.read_csv(ENTITY_RESOLVED_PATH, parse_dates=['date'])
registry_report_df = pd.read_csv(ENTITY_REGISTRY_PATH)
raw_api_response_lines = sum(1 for _ in RAW_RESPONSES_PATH.open('r', encoding='utf-8'))

quality_summary_report = manifest['quality_summary']
row_counts_report = manifest['row_counts']
date_range_report = manifest['date_range']

collection_scope = pd.DataFrame([
    {'item': 'notebook', 'value': 'notebooks/01_data_collection.ipynb'},
    {'item': 'dataset_name', 'value': manifest['dataset_name']},
    {'item': 'date_start', 'value': date_range_report['start']},
    {'item': 'date_end', 'value': date_range_report['end']},
    {'item': 'frequency', 'value': date_range_report['frequency']},
    {'item': 'canonical_entities', 'value': manifest['canonical_entities']},
    {'item': 'sources', 'value': len(manifest['sources'])},
    {'item': 'raw_api_requests', 'value': row_counts_report['raw_api_requests']},
    {'item': 'source_observation_rows', 'value': row_counts_report['source_observations']},
    {'item': 'rows_per_source', 'value': quality_summary_report['expected_rows_per_source']},
    {'item': 'entity_resolved_rows', 'value': row_counts_report['entity_resolved_rows']},
    {'item': 'entity_registry_rows', 'value': row_counts_report['entity_registry_rows']},
    {'item': 'expected_daily_dates_per_city', 'value': quality_summary_report['expected_daily_dates']},
    {'item': 'secret_policy', 'value': manifest['secret_policy']},
])

source_summary = (
    source_report_df
    .groupby('source', as_index=False)
    .agg(
        rows=('source', 'size'),
        cities=('entity_id', 'nunique'),
        date_min=('date', 'min'),
        date_max=('date', 'max'),
        precip_mean_mm_day=('precipitation_mm', 'mean'),
        precip_min_mm_day=('precipitation_mm', 'min'),
        precip_max_mm_day=('precipitation_mm', 'max'),
    )
)
source_summary['date_min'] = source_summary['date_min'].dt.date.astype(str)
source_summary['date_max'] = source_summary['date_max'].dt.date.astype(str)

downloaded_sources = pd.DataFrame([
    {
        'source_name': source['name'],
        'source_code': 'NASA_POWER_DAILY' if 'NASA' in source['name'] else 'OPEN_METEO_ARCHIVE',
        'base_url': source['base_url'],
        'requires_api_key': source['requires_api_key'],
        'request_count': row_counts_report['raw_api_requests'] // len(manifest['sources']),
        'rows': int(source_summary.loc[source_summary['source'].eq('NASA_POWER_DAILY' if 'NASA' in source['name'] else 'OPEN_METEO_ARCHIVE'), 'rows'].iloc[0]),
        'documentation': source['documentation'],
    }
    for source in manifest['sources']
])

variable_mapping = pd.DataFrame([
    {'standard_column': 'precipitation_mm', 'unit': 'mm/day', 'nasa_power_parameter': 'PRECTOTCORR', 'open_meteo_variable': 'precipitation_sum', 'standardization_note': 'Daily precipitation amount.'},
    {'standard_column': 'temp_mean_c', 'unit': 'C', 'nasa_power_parameter': 'T2M', 'open_meteo_variable': 'temperature_2m_mean', 'standardization_note': 'Already Celsius in both sources.'},
    {'standard_column': 'temp_max_c', 'unit': 'C', 'nasa_power_parameter': 'T2M_MAX', 'open_meteo_variable': 'temperature_2m_max', 'standardization_note': 'Already Celsius in both sources.'},
    {'standard_column': 'temp_min_c', 'unit': 'C', 'nasa_power_parameter': 'T2M_MIN', 'open_meteo_variable': 'temperature_2m_min', 'standardization_note': 'Already Celsius in both sources.'},
    {'standard_column': 'relative_humidity_pct', 'unit': 'percent', 'nasa_power_parameter': 'RH2M', 'open_meteo_variable': 'relative_humidity_2m_mean', 'standardization_note': 'Daily mean relative humidity.'},
    {'standard_column': 'wind_speed_ms', 'unit': 'm/s', 'nasa_power_parameter': 'WS2M', 'open_meteo_variable': 'wind_speed_10m_mean', 'standardization_note': 'Open-Meteo is converted from km/h to m/s by dividing by 3.6.'},
    {'standard_column': 'surface_pressure_kpa', 'unit': 'kPa', 'nasa_power_parameter': 'PS', 'open_meteo_variable': 'surface_pressure_mean', 'standardization_note': 'Open-Meteo is converted from hPa to kPa by dividing by 10.'},
])

output_files = pd.DataFrame([
    {'file_path': str(RAW_RESPONSES_PATH.relative_to(PROJECT_ROOT)), 'meaning': 'Raw JSON API responses and request URLs without secrets.', 'row_count_or_lines': raw_api_response_lines, 'primary_use': 'Reproducibility and audit trail.'},
    {'file_path': str(SOURCE_OBSERVATIONS_PATH.relative_to(PROJECT_ROOT)), 'meaning': 'Long-form source observations before wide merge.', 'row_count_or_lines': len(source_report_df), 'primary_use': 'Source-level quality checks and source agreement analysis.'},
    {'file_path': str(ENTITY_RESOLVED_PATH.relative_to(PROJECT_ROOT)), 'meaning': 'Wide entity-date table with NASA and Open-Meteo side by side.', 'row_count_or_lines': len(resolved_report_df), 'primary_use': 'Main raw dataset for Notebook 02 and later preprocessing.'},
    {'file_path': str(ENTITY_REGISTRY_PATH.relative_to(PROJECT_ROOT)), 'meaning': 'Canonical entity registry and source-coordinate matching metadata.', 'row_count_or_lines': len(registry_report_df), 'primary_use': 'Entity resolution audit.'},
    {'file_path': str(MANIFEST_PATH.relative_to(PROJECT_ROOT)), 'meaning': 'Manifest with metadata, row counts, hashes, and quality summary.', 'row_count_or_lines': 1, 'primary_use': 'Dataset documentation and reproducibility.'},
])

quality_summary_table = pd.DataFrame([
    {'quality_check': 'expected_daily_dates', 'result': 'pass', 'value_or_note': quality_summary_report['expected_daily_dates']},
    {'quality_check': 'expected_rows_per_source', 'result': 'pass', 'value_or_note': quality_summary_report['expected_rows_per_source']},
    {'quality_check': 'NASA_POWER_DAILY_rows', 'result': 'pass', 'value_or_note': quality_summary_report['actual_rows_by_source'].get('NASA_POWER_DAILY')},
    {'quality_check': 'OPEN_METEO_ARCHIVE_rows', 'result': 'pass', 'value_or_note': quality_summary_report['actual_rows_by_source'].get('OPEN_METEO_ARCHIVE')},
    {'quality_check': 'source_duplicate_rows', 'result': 'pass', 'value_or_note': quality_summary_report['source_duplicate_rows']},
    {'quality_check': 'resolved_duplicate_rows', 'result': 'pass', 'value_or_note': quality_summary_report['resolved_duplicate_rows']},
    {'quality_check': 'negative_precipitation_rows', 'result': 'pass', 'value_or_note': quality_summary_report['negative_precipitation_rows']},
    {'quality_check': 'missing_standard_variables', 'result': 'pass', 'value_or_note': '0 percent for both sources'},
    {'quality_check': 'entity_resolution_threshold', 'result': 'pass', 'value_or_note': manifest['entity_resolution']['method']},
])

pipeline_steps = pd.DataFrame([
    {'step': 1, 'operation': 'Define canonical locations', 'output_or_effect': 'Creates 12 stable city-level entity_id values.'},
    {'step': 2, 'operation': 'Build API URLs', 'output_or_effect': 'Creates NASA POWER and Open-Meteo no-key daily archive requests.'},
    {'step': 3, 'operation': 'Fetch JSON payloads', 'output_or_effect': 'Downloads 24 raw API responses.'},
    {'step': 4, 'operation': 'Parse source variables', 'output_or_effect': 'Maps each source to the shared standard schema.'},
    {'step': 5, 'operation': 'Standardize units', 'output_or_effect': 'Converts Open-Meteo wind speed and pressure units.'},
    {'step': 6, 'operation': 'Resolve entities', 'output_or_effect': 'Maps source coordinates to nearest canonical entity within 50 km.'},
    {'step': 7, 'operation': 'Save raw responses', 'output_or_effect': 'Writes raw JSONL audit file.'},
    {'step': 8, 'operation': 'Save source observations', 'output_or_effect': 'Writes long-form source observations CSV.'},
    {'step': 9, 'operation': 'Build resolved table', 'output_or_effect': 'Merges NASA and Open-Meteo by entity_id and date.'},
    {'step': 10, 'operation': 'Compute immediate source fields', 'output_or_effect': 'Adds two-source precipitation mean and absolute difference.'},
    {'step': 11, 'operation': 'Run quality checks', 'output_or_effect': 'Checks duplicate rows, negative rainfall, missingness, and expected counts.'},
    {'step': 12, 'operation': 'Write manifest', 'output_or_effect': 'Writes metadata, hashes, source info, and quality summary.'},
])

entity_columns = [
    'entity_id',
    'country',
    'location_name',
    'canonical_latitude',
    'canonical_longitude',
    'nasa_power_distance_to_canonical_km',
    'open_meteo_distance_to_canonical_km',
]
canonical_entities = registry_report_df[entity_columns].copy()

for table, filename in [
    (collection_scope, '01_collection_scope.csv'),
    (downloaded_sources, '02_downloaded_sources.csv'),
    (variable_mapping, '03_variable_mapping.csv'),
    (output_files, '04_output_files.csv'),
    (quality_summary_table, '05_quality_summary.csv'),
    (pipeline_steps, '06_pipeline_steps.csv'),
    (canonical_entities, '07_canonical_entities.csv'),
    (source_summary, '08_source_summary.csv'),
]:
    table.to_csv(REPORT_TABLE_DIR / filename, index=False)

report_text = f'''# 01 Data Collection Report

## Purpose
Notebook `notebooks/01_data_collection.ipynb` builds the raw multi-source rainfall dataset for Southeast Asia. It is a data collection and entity-resolution notebook, not a modeling notebook.

## What It Downloads
- Source 1: NASA POWER Daily API.
- Source 2: Open-Meteo Historical Weather API.
- Both sources require no API key.
- Date range: `{date_range_report['start']}` to `{date_range_report['end']}`.
- Frequency: `{date_range_report['frequency']}`.
- Locations: {manifest['canonical_entities']} canonical Southeast Asia city entities.
- Total API requests: {row_counts_report['raw_api_requests']}.

The collected variables are daily precipitation, temperature, relative humidity, wind speed, and surface pressure. Units are standardized into `mm/day`, `C`, `percent`, `m/s`, and `kPa`.

## Entity Resolution
Each source-returned coordinate is mapped to the nearest canonical city coordinate within 50 km using the Haversine distance. After that, NASA POWER and Open-Meteo records are merged by `entity_id + date`.

## Output Files
- `{RAW_RESPONSES_PATH.relative_to(PROJECT_ROOT)}`: raw JSONL API audit trail.
- `{SOURCE_OBSERVATIONS_PATH.relative_to(PROJECT_ROOT)}`: long-form source observations.
- `{ENTITY_RESOLVED_PATH.relative_to(PROJECT_ROOT)}`: wide entity-date table with both sources side by side.
- `{ENTITY_REGISTRY_PATH.relative_to(PROJECT_ROOT)}`: entity-resolution registry.
- `{MANIFEST_PATH.relative_to(PROJECT_ROOT)}`: dataset manifest and hashes.

## Row Counts
- Raw API requests: {row_counts_report['raw_api_requests']}.
- Source observation rows: {row_counts_report['source_observations']}.
- Rows per source: {quality_summary_report['expected_rows_per_source']}.
- Entity-resolved rows: {row_counts_report['entity_resolved_rows']}.
- Entity registry rows: {row_counts_report['entity_registry_rows']}.

Formula: `12 cities x 2,192 daily dates = 26,304 rows per source`.

## Quality Result
- Source duplicate rows: {quality_summary_report['source_duplicate_rows']}.
- Entity-date duplicate rows after resolution: {quality_summary_report['resolved_duplicate_rows']}.
- Negative precipitation rows: {quality_summary_report['negative_precipitation_rows']}.
- Missing rate for standard variables: 0 percent for both sources.

## Interpretation
Notebook 01 does not decide which source is more correct. It only collects and aligns NASA POWER and Open-Meteo. Because there is no independent station/rain-gauge ground truth here, source disagreement must be analyzed in later notebooks.

## Report Tables
See `reports/01_data_collection_report/tables/` for compact CSV documentation tables.
'''

(REPORT_DIR / 'DATA_COLLECTION_REPORT.md').write_text(report_text, encoding='utf-8')

print('Notebook 01 report written to:', REPORT_DIR)
print('Main report:', REPORT_DIR / 'DATA_COLLECTION_REPORT.md')
print('Tables:', REPORT_TABLE_DIR)

Notebook 01 report written to: D:\DS\Project\reports\01_data_collection_report
Main report: D:\DS\Project\reports\01_data_collection_report\DATA_COLLECTION_REPORT.md
Tables: D:\DS\Project\reports\01_data_collection_report\tables
